In [1]:
import torch, torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython import display
%matplotlib inline

# ... 数据生成和模型定义部分保持不变 ...

# 准备网格
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

losses = []
epochs = 500
update_interval = 5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))

for epoch in range(epochs):
    # 训练一步
    model.train()
    optimizer.zero_grad()
    pred = model(X)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    # 每 update_interval 更新图表
    if epoch % update_interval == 0 or epoch == epochs-1:
        # 左图：损失曲线
        ax1.clear()
        ax1.plot(losses, 'b-', linewidth=1)
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.set_title('Training Loss')
        ax1.set_xlim(0, epochs)
        ax1.set_ylim(0, max(losses)*1.1 if losses else 1)

        # 右图：决策边界
        model.eval()
        with torch.no_grad():
            Z = model(grid).numpy().reshape(xx.shape)
        ax2.clear()
        ax2.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.6)
        ax2.scatter(X[:,0], X[:,1], c=y.squeeze(), cmap='RdBu', edgecolors='k')
        ax2.set_xlim(x_min, x_max); ax2.set_ylim(y_min, y_max)
        ax2.set_title(f'Decision Boundary (Epoch {epoch})')

        # 用 IPython 的 display 更新整个 figure
        display.clear_output(wait=True)
        display.display(fig)

plt.close()

NameError: name 'X' is not defined

$$
a = \sigma \left( \sum_{i=1}^{n} w_i x_i + b \right)
$$

$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}\\
a^{[l]} = \sigma \left( z^{[l]} \right)
$$

$$
\delta^{[l]} = \frac{\partial J}{\partial z^{[l]}} \in \mathbb{R}^{n_l}
$$

$$
\delta^{[l]} = \frac{\partial J}{\partial z^{[l]}} = \frac{\partial J}{\partial z^{[l+1]}} \cdot \frac{\partial z^{[l+1]}}{\partial a^{[l]}} \cdot \frac{\partial a^{[l]}}{\partial z^{[l]}}
$$

$$
\delta^{[l]} = \underbrace{\left( W^{[l+1]} \right)^T \delta^{[l+1]}}_{\text{误差加权回传}} \odot \underbrace{\sigma'\left( z^{[l]} \right)}_{\text{本地梯度}}
$$

$$
\frac{\partial J}{\partial b^{[l]}} = \delta^{[l]}
$$

$$
\frac{\partial J}{\partial W^{[l]}} = \delta^{[l]} \left( a^{[l-1]} \right)^T
$$

$$
W^{[l]} := W^{[l]} - \eta \frac{\partial J}{\partial W^{[l]}}, \quad b^{[l]} := b^{[l]} - \eta \frac{\partial J}{\partial b^{[l]}}
$$

In [ ]:
import numpy as np

#激活函数与导数
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1-s)
#损失函数：BCE
def compute_loss(y_true,y_pred):
    y_pred = np.clip(y_pred,1e-12,1-1e-12)
    return -np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))

class NeuralNetwork:
    def __init__(self,layer_sizes):
       self.num_layers = len(layer_sizes) - 1 
       self.weights = []
       self.biases = []
       for i in range(self.num_layers):
           w = np.random.randn(layer_sizes[i+1],layer_sizes[i]) * 0.1
           b = np.random.randn(layer_sizes[i+1],1) * 0.1
    def forward(self,x):
        a = x 
        activations = [a]
        zs = []
        for w,b in zip(self.weights,self.biases):
            z = np.dot(w,a) + b
            zs.append(z)
            a = sigmoid(z)
            activations.append(a)
        return activations,zs
    def backward(self,x,y,activations,zs,learning_rate):
        m = x.shape[1]
        delta = activations[-1] - y
        for l in range(self.num_layers - 1,-1,-1):
            dW = (1 / m) * np.dot(delta, activations[l].T)
            db = (1 / m) * np.sum(delta, axis=1, keepdims=True)
            self.weights[l] -= learning_rate * dW
            self.biases[l]  -= learning_rate * db
            if l > 0:
                delta = np.dot(self.weights[l].T, delta) * sigmoid_prime(zs[l-1])

           


